This notebooks based on https://www.kaggle.com/code/kurisew/lb0-855-openvino-multithread-tta-ensemble-infer <br>
By the ensemble order different with me : https://www.kaggle.com/code/hideyukizushi/bird25-weightedblend-nfnet-convnextv2-lb-860 <br>
I found the best ensemble order, it will cost nearly 1 hours 5 minutes to get a stable score 0.874. <br>
Thanks to the great notebooks : https://www.kaggle.com/code/i2nfinit3y/bird2025-single-sed-model-inference-lb-0-857 <br>
                                https://www.kaggle.com/code/myso1987/post-processing-with-power-adjustment-for-low-rank <br>
The inference order is Single SED -> Openvino_model -> 3-fold SED <br>
There is still a problem in 3-fold SED, changing mel will long the time significantly, maybe we need to use librosa to replace it. <br>

note: You can get better score on LB, just by adjusting hyperparameters. (0.88 or higher) But I think it will overfit the LB and that's why I call this is a baseline.

By watching last year of this competition, shakeup is a great problem. I want to find a way to avoid this. But there is no test data for us to make reliable CV. Really hoping we will get few test files next year.

In [1]:
##open vino installation..
! python -m pip install --no-index --find-links=../input/openvino-wheels -r ../input/openvino-wheels/requirements.txt

Looking in links: ../input/openvino-wheels
Processing /kaggle/input/openvino-wheels/openvino_dev-2024.6.0-17404-py3-none-any.whl (from openvino-dev[onnx]==2024.6.0->-r ../input/openvino-wheels/requirements.txt (line 1))
Processing /kaggle/input/openvino-wheels/networkx-3.1-py3-none-any.whl (from openvino-dev==2024.6.0->openvino-dev[onnx]==2024.6.0->-r ../input/openvino-wheels/requirements.txt (line 1))
Processing /kaggle/input/openvino-wheels/openvino_telemetry-2025.1.0-py3-none-any.whl (from openvino-dev==2024.6.0->openvino-dev[onnx]==2024.6.0->-r ../input/openvino-wheels/requirements.txt (line 1))
Processing /kaggle/input/openvino-wheels/openvino-2024.6.0-17404-cp311-cp311-manylinux2014_x86_64.whl (from openvino-dev==2024.6.0->openvino-dev[onnx]==2024.6.0->-r ../input/openvino-wheels/requirements.txt (line 1))
Processing /kaggle/input/openvino-wheels/fastjsonschema-2.17.1-py3-none-any.whl (from openvino-dev[onnx]==2024.6.0->-r ../input/openvino-wheels/requirements.txt (line 1))
  Att

In [2]:

import os
import gc
import warnings
import logging
import time
import math
import cv2
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch 
import torchaudio
import torchaudio.transforms as AT
from typing import Union, List, Dict, Any 
import concurrent.futures
import itertools
import openvino.runtime as ov
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)

class CFG:
    # General Paths
    data_base_path = "/kaggle/input/birdclef-2025/"#/kaggle/input/birdclef-2025/sample_submission.csv
    train_audio_dir = os.path.join(data_base_path, "train_audio/")
    test_soundscapes_dir = os.path.join(data_base_path, "test_soundscapes/")
    train_soundscapes_dir = os.path.join(data_base_path, "train_soundscapes/")
    sample_submission_csv = os.path.join(data_base_path, "sample_submission.csv")
    taxonomy_csv = os.path.join(data_base_path, "taxonomy.csv")

    # --- Stage 1: NFNet SED Models (OpenVINO) ---
    # !!! REPLACE 'your-openvino-models-dataset' and 'birdclef-2025-sed-models-p' as needed !!!
    nfnet_model_ir_base_path = "/kaggle/input/open-vino-models/" 
    nfnet_sed_fold0_xml = os.path.join(nfnet_model_ir_base_path, "nfnet_sed_fold0_openvino_batch12/nfnet_sed_fold0_batch12.xml")
    nfnet_sed_fold1_xml = os.path.join(nfnet_model_ir_base_path, "nfnet_sed_fold1_openvino_batch12/nfnet_sed_fold1_batch12.xml")
    nfnet_sed_fold2_xml = os.path.join(nfnet_model_ir_base_path, "nfnet_sed_fold2_openvino_batch12/nfnet_sed_fold2_batch12.xml")
    
    original_nfnet_ckpt_base_path = '/kaggle/input/birdclef-2025-sed-models-p/' 
    original_nfnet_ckpt_paths = [
        os.path.join(original_nfnet_ckpt_base_path, 'sed0.pth'),
        os.path.join(original_nfnet_ckpt_base_path, 'sed1.pth'),
        os.path.join(original_nfnet_ckpt_base_path, 'sed2.pth')
    ]
    nfnet_checkpoint_config = None 
    
    nfnet_fallback_config = {
        'SR': 32000, 'n_mels': 128, 'hop_length': 512, 'f_min': 40, 
        'f_max': 15000, 'n_fft': 1024, 'win_length': 1024, 
        'wav_sec': 5, 'input_channels_for_encoder': 3 ,'openvino_input_channels': 1
    }

    # --- Stage 2: Seresnext Model (OpenVINO - multi-part) ---
    # !!! REPLACE paths as needed !!!
    seresnext_model_ir_base_path = "/kaggle/input/open-vino-models/" 
    srx_extract_feature_xml = os.path.join(seresnext_model_ir_base_path, "seresnext_openvino_ir/srx_extract_feature_ov/srx_extract_feature.xml")
    srx_att_block_att_xml = os.path.join(seresnext_model_ir_base_path, "seresnext_openvino_ir/srx_att_block_att_ov/srx_att_block_att.xml")
    srx_att_block_cla_xml = os.path.join(seresnext_model_ir_base_path, "seresnext_openvino_ir/srx_att_block_cla_ov/srx_att_block_cla.xml")
    
    original_seresnext_ckpt_path = '/kaggle/input/bird2025-sed-ckpt/sedmodel.pth'
    srx_checkpoint_config = None 
    
    srx_fallback_config = {
        'SR': 32000, 'n_mels': 128, 'hop_length': 512, 'f_min': 20, 
        'f_max': 16000, 'n_fft': 2048, 'normal': 80, 
        'in_channels': 1, 'duration_train': 10, 
        'infer_duration': 5, 'apply_image_delta': False, 'top_db': 80
    }
    srx_tta_delta = 2

    # --- Stage 3: EfficientNet B0 + RegNetY_008 (OpenVINO) ---
    # !!! REPLACE 'your-openvino-models-dataset' as needed !!!
    regenety_base_path="/kaggle/input/regnety_008/pytorch/default/1/"
    effnet_regnet_base_path = "/kaggle/input/efficientnet_b0/pytorch/openvino/1/" 
    efficientnet_b0_xml = os.path.join(effnet_regnet_base_path, "efficientnet_b0.xml")
    regnety_008_xml = os.path.join(regenety_base_path, "regnety_008.xml")
    
    sr_effreg = 32000; wav_sec_effreg = 5; n_fft_effreg = 1024; hop_length_effreg = 256 
    n_mels_effreg = 128; f_min_effreg = 48; f_max_effreg = 15000
    effreg_target_shape = (256,256); effreg_input_channels = 1 

    # General
    num_workers = 4 
    ensemble_weights = [0.05, 0.6, 0.35] # TUNE! [nfnet, seresnext, effreg]
    
    # Debug
    debug = True 
    debug_file_count = 5

cfg = CFG()

def load_checkpoint_config_from_path(ckpt_path: str, model_name_for_log: str) -> Union[Dict, None]:
    # ... (This function definition remains the same as in the previous response) ...
    if os.path.exists(ckpt_path):
        print(f"Loading {model_name_for_log} original checkpoint's config from: {ckpt_path}")
        try:
            checkpoint_data = torch.load(ckpt_path, map_location='cpu', weights_only=False) 
            if 'cfg' in checkpoint_data:
                loaded_cfg_obj = checkpoint_data['cfg']
                config_dict = None
                if not isinstance(loaded_cfg_obj, dict): 
                    try: config_dict = vars(loaded_cfg_obj)
                    except TypeError:
                        print(f"Warning: {model_name_for_log} 'cfg' is object, not directly vars() convertible. Trying known attributes.")
                        known_attrs = ['SR', 'n_mels', 'hop_length', 'f_min', 'f_max', 'n_fft', 
                                       'normal', 'in_channels', 'duration_train', 'infer_duration', 
                                       'apply_image_delta', 'top_db', 'num_classes', 
                                       'win_length', 'wav_sec', 'model_name', 
                                       'target_duration', 'train_duration', 'input_channels_for_encoder'] 
                        config_dict = {attr: getattr(loaded_cfg_obj, attr) for attr in known_attrs if hasattr(loaded_cfg_obj, attr)}
                        if not config_dict: return None
                else: config_dict = loaded_cfg_obj
                if config_dict:
                    print(f"Successfully loaded {model_name_for_log} config. Keys: {list(config_dict.keys())}")
                    return config_dict
                else: return None
            else: print(f"Warning: 'cfg' key not found in {model_name_for_log} checkpoint ({ckpt_path})."); return None
        except Exception as e: print(f"Error loading {model_name_for_log} config from {ckpt_path}: {e}"); return None
    else: print(f"Warning: Original {model_name_for_log} checkpoint {ckpt_path} not found."); return None

# Load NFNet Config
if cfg.original_nfnet_ckpt_paths and os.path.exists(cfg.original_nfnet_ckpt_paths[0]):
    cfg.nfnet_checkpoint_config = load_checkpoint_config_from_path(cfg.original_nfnet_ckpt_paths[0], "NFNet Fold 0")
if cfg.nfnet_checkpoint_config is None:
    print("Using fallback parameters for NFNet.")
    cfg.nfnet_checkpoint_config = cfg.nfnet_fallback_config.copy()

# Load SeresNext Config
cfg.srx_checkpoint_config = load_checkpoint_config_from_path(cfg.original_seresnext_ckpt_path, "SeresNext")
if cfg.srx_checkpoint_config is None:
    print("Using fallback parameters for SeresNext.")
    cfg.srx_checkpoint_config = cfg.srx_fallback_config.copy()

taxonomy_df = pd.read_csv(cfg.taxonomy_csv)
species_ids = taxonomy_df['primary_label'].tolist()
num_classes = len(species_ids) 
try: class_labels_global = sorted(os.listdir(cfg.train_audio_dir)) 
except FileNotFoundError: class_labels_global = species_ids
print(f"Global Number of classes: {num_classes}; Debug mode: {cfg.debug}")
core = ov.Core()


Loading NFNet Fold 0 original checkpoint's config from: /kaggle/input/birdclef-2025-sed-models-p/sed0.pth
Using fallback parameters for NFNet.
Loading SeresNext original checkpoint's config from: /kaggle/input/bird2025-sed-ckpt/sedmodel.pth
Successfully loaded SeresNext config. Keys: ['seed', 'debug', 'print_freq', 'num_workers', 'stage', 'OUTPUT_DIR', 'train_datadir', 'train_csv', 'label_csv', 'test_soundscapes', 'submission_csv', 'taxonomy_csv', 'class_sample_count', 'model_name', 'pretrained', 'in_channels', 'img_size', 'target_duration', 'SR', 'n_fft', 'n_mels', 'f_min', 'f_max', 'hop_length', 'device', 'epochs', 'batch_size', 'criterion', 'use_weights', 'n_fold', 'selected_folds', 'optimizer', 'lr', 'weight_decay', 'scheduler', 'min_lr', 'warmup_epo', 'T_max', 'mixup_prob', 'mixup_double', 'mix_beta', 'mix_beta2', 'mixup2_prob', 'cutmix_beta', 'cutmix_prob', 'sumix_max_percent', 'sumix_min_percent', 'mixup', 'mixup2', 'cutmix', 'sumix', 'normal', 'valid_duration', 'duration_train'

In [3]:

# --- Utility: Post-processing (apply_power_to_low_ranked_cols) - Unchanged ---
def apply_power_to_low_ranked_cols(p: np.ndarray, top_k: int = 30, exponent: Union[int, float] = 2, inplace: bool = True) -> np.ndarray:
    if not inplace: p = p.copy()
    if p.ndim == 1: 
        p_temp = np.expand_dims(p, axis=0)
        tail_cols = np.argsort(-p_temp.max(axis=0))[top_k:]
        p[tail_cols] = p[tail_cols] ** exponent
    elif p.ndim == 2: 
        tail_cols = np.argsort(-p.max(axis=0))[top_k:]
        p[:, tail_cols] = p[:, tail_cols] ** exponent
    return p

# --- Utility: Temporal Smoothing (smooth_submission) - Unchanged ---
def smooth_submission(sub_df_to_smooth: pd.DataFrame) -> pd.DataFrame:
    print("Smoothing submission predictions...")
    cols_to_smooth = sub_df_to_smooth.columns[1:]
    groups = sub_df_to_smooth['row_id'].str.rsplit('_', n=1).str[0].values
    unique_groups = np.unique(groups)
    smoothed_df = sub_df_to_smooth.copy()
    for group in unique_groups:
        idx = np.where(groups == group)[0]
        sub_group = smoothed_df.iloc[idx].copy() 
        predictions = sub_group[cols_to_smooth].values
        if predictions.shape[0] <= 1: continue
        new_predictions = predictions.copy()
        new_predictions[0] = (predictions[0] * 0.8) + (predictions[1] * 0.2)
        new_predictions[-1] = (predictions[-1] * 0.8) + (predictions[-2] * 0.2)
        for i in range(1, predictions.shape[0] - 1):
            new_predictions[i] = (predictions[i-1] * 0.2) + (predictions[i] * 0.6) + (predictions[i+1] * 0.2)
        smoothed_df.iloc[idx, 1:] = new_predictions
    print(f"Smoothing complete.")
    return smoothed_df

# --- TORCHAUDIO-BASED PREPROCESSING (Used for NFNet - Stage 1) ---
# This function will use parameters from the passed `nfnet_loaded_config`

# This is part of what Cell 2 should look like.
# Other functions like apply_power_to_low_ranked_cols, smooth_submission,
# SeresNext preprocessing, Librosa preprocessing will also be in Cell 2.

# --- TORCHAUDIO-BASED PREPROCESSING (Used for NFNet - Stage 1) ---
# This function will use parameters from the passed `nfnet_loaded_config`

def normalize_std_torchaudio(spec, eps=1e-6): # Helper, ensure this is defined in Cell 2
    mean = torch.mean(spec)
    std = torch.std(spec)
    return torch.where(std < eps, spec - mean, (spec - mean) / (std + eps))

def preprocess_audio_nfnet_dynamic_config(filepath: str, nfnet_loaded_config: dict) -> Union[np.ndarray, None]:
    # Extract params from the loaded nfnet_config, with fallbacks from global CFG
    _sr = nfnet_loaded_config.get('SR', cfg.nfnet_fallback_config['SR'])
    _wav_sec = nfnet_loaded_config.get('wav_sec', cfg.nfnet_fallback_config['wav_sec'])
    _n_fft = nfnet_loaded_config.get('n_fft', cfg.nfnet_fallback_config['n_fft'])
    _win_length = nfnet_loaded_config.get('win_length', cfg.nfnet_fallback_config['win_length'])
    _hop_length = nfnet_loaded_config.get('hop_length', cfg.nfnet_fallback_config['hop_length'])
    _f_min = nfnet_loaded_config.get('f_min', cfg.nfnet_fallback_config['f_min'])
    _f_max = nfnet_loaded_config.get('f_max', cfg.nfnet_fallback_config['f_max'])
    _n_mels = nfnet_loaded_config.get('n_mels', cfg.nfnet_fallback_config['n_mels'])
    # 'openvino_input_channels' from nfnet_loaded_config will determine final channel count.
    # Default to 1 if not specified, as the error suggests the OV model expects 1 channel.
    _openvino_input_channels = nfnet_loaded_config.get('openvino_input_channels', 1)


    _mel_transform_current_nfnet = AT.MelSpectrogram(
        sample_rate=_sr, n_fft=_n_fft, win_length=_win_length,
        hop_length=_hop_length, center=True, f_min=_f_min, f_max=_f_max,
        pad_mode="reflect", power=2.0, norm='slaney', n_mels=_n_mels, mel_scale="htk",
    )
    
    try: 
        waveform, sr_orig_load = torchaudio.load(filepath, backend="soundfile")
    except Exception as e: 
        print(f"Error loading {filepath}: {e}")
        return None
        
    if sr_orig_load != _sr: 
        waveform = torchaudio.functional.resample(waveform, sr_orig_load, _sr)
    
    len_wav = waveform.shape[1]
    if waveform.ndim > 1 and waveform.shape[0] > 1: 
        waveform = waveform[0,:].reshape(1, len_wav) # Ensure [1, samples]
    elif waveform.ndim == 1: 
        waveform = waveform.reshape(1, len_wav) # Ensure [1, samples]
        
    segments_processed_np = []
    num_full_segments = len_wav // (_sr * _wav_sec)

    for i in range(num_full_segments):
        start_sample = i * _sr * _wav_sec
        end_sample = start_sample + (_sr * _wav_sec)
        waveform_chunk = waveform[:, start_sample:end_sample] # Shape [1, samples_in_chunk]
        
        melspec = _mel_transform_current_nfnet(waveform_chunk) # Output: [1, n_mels, time_frames]
        melspec = torch.log(melspec + 1e-6)
        melspec = normalize_std_torchaudio(melspec) # Output: [1, n_mels, time_frames]
        
        # --- CRITICAL CHANGE AREA START ---
        # The OpenVINO model error indicated it expects 1 channel.
        # The melspec tensor at this point is already [1, n_mels, time_frames],
        # which corresponds to [Channels=1, Height=n_mels, Width=time_frames].
        # So, we don't need to explicitly change the number of channels if _openvino_input_channels is 1.
        # If for some reason the OpenVINO model *did* expect 3 channels (contrary to error),
        # then the .repeat() logic would be used.
        
        if _openvino_input_channels == 3:
            # This case would be if the OV model expects 3 channels, but error suggests 1.
            # For safety, keeping this logic if config explicitly asks for 3.
            if melspec.shape[0] == 1: # If input is mono-channel like [1, H, W]
                processed_melspec_for_ov = melspec.repeat(_openvino_input_channels, 1, 1) # -> [3, H, W]
            else: # If melspec was already multi-channel somehow and not 1.
                processed_melspec_for_ov = melspec 
        elif _openvino_input_channels == 1:
            # Ensure it's [1, H, W] - which melspec already is after AT.MelSpectrogram
            processed_melspec_for_ov = melspec # melspec is [1, n_mels, time_frames]
        else:
            print(f"Warning: Unsupported _openvino_input_channels: {_openvino_input_channels}. Defaulting to 1 channel output.")
            processed_melspec_for_ov = melspec # Default to the 1-channel melspec

        # The output of this function will be a list of numpy arrays,
        # where each array is [C_model_expects, H, W].
        # `np.vstack` later will make it [NumSegments, C_model_expects, H, W].
        # However, if each item is already [1,C,H,W] from PyTorch, then vstack is fine for OV's NCHW
        # No, `processed_melspec_for_ov` here is a single segment's spec.
        # If `processed_melspec_for_ov` is [C, H, W] then `vstack` will make it [NumSeg, C, H, W]
        # If `processed_melspec_for_ov` is [1, H, W] (after squeeze(0) for channel),
        # and C is 1, then vstack makes it [NumSeg, H, W]. Then expand_dims in infer needed.

        # Let's ensure each item in segments_processed_np is shaped [C,H,W]
        # melspec from AT.MelSpectrogram is [1, n_mels, time_frames] (effectively Batch=1, C=1, H, W for the transform)
        # The OpenVINO model expects C=1. So we need each segment to be [1, n_mels, time_frames]
        
        # The melspec variable is already [1, n_mels, time_frames]
        # This is the correct [C,H,W] for a single segment if C=1
        segments_processed_np.append(melspec.cpu().numpy()) 
        # --- CRITICAL CHANGE AREA END ---
        
    if not segments_processed_np: 
        return None
    
    # segments_processed_np is a list of arrays, each [1, n_mels, time_frames]
    # np.concatenate along axis 0 will result in [num_segments, n_mels, time_frames] if we squeeze before.
    # np.array will result in [num_segments, 1, n_mels, time_frames]
    return np.array(segments_processed_np) if segments_processed_np else None

# ... (Rest of Cell 2: SeresNext preprocessing, Librosa preprocessing, etc. should be here) ...   

# --- PYTORCH-BASED SPECTROGRAM TRANSFORMATION FOR SERESNEXT (Stage 2) ---
# (compute_deltas_for_transform and image_delta_for_transform definitions as in previous response)
def compute_deltas_for_transform(specgram: torch.Tensor, win_length: int = 5, mode: str = "replicate") -> torch.Tensor:
    device = specgram.device; dtype = specgram.dtype; 
    n = (win_length - 1) // 2
    if n < 1: return torch.zeros_like(specgram) 
    denom = n * (n + 1) * (2 * n + 1) / 3
    if specgram.dim() == 2: specgram_reshaped_for_conv = specgram.unsqueeze(0).unsqueeze(0)
    elif specgram.dim() == 3: specgram_reshaped_for_conv = specgram.unsqueeze(0)
    else: specgram_reshaped_for_conv = specgram
    kernel_vals = torch.arange(-n, n + 1, 1, device=device, dtype=dtype) 
    kernel = kernel_vals.view(1, 1, -1) 
    B, C_spec, Freq, Time = specgram_reshaped_for_conv.shape; delta_output_list = []
    for b_idx in range(B):
        batch_slice_list = []
        for c_idx in range(C_spec):
            current_spec_slice = specgram_reshaped_for_conv[b_idx, c_idx, :, :] 
            padded_slice = F.pad(current_spec_slice, (n, n), mode=mode) 
            delta_freq_rows = F.conv1d(padded_slice.unsqueeze(1), kernel.repeat(Freq,1,1), groups=Freq, bias=None) / denom
            batch_slice_list.append(delta_freq_rows.squeeze(1)) 
        if C_spec > 1 and batch_slice_list: delta_output_list.append(torch.stack(batch_slice_list, dim=0)) 
        elif batch_slice_list: delta_output_list.append(batch_slice_list[0].unsqueeze(0))             
    if not delta_output_list: return torch.empty_like(specgram_reshaped_for_conv)
    final_delta = torch.stack(delta_output_list, dim=0)
    if final_delta.ndim == 4 and final_delta.shape[0] == B and final_delta.shape[1] == C_spec and final_delta.shape[2] == Freq and final_delta.shape[3] == Time:
        pass # Shape is already correct
    else: # Attempt to reshape if dimensions are off, might indicate deeper issue
        try: final_delta = final_delta.view(B,C_spec,Freq,Time)
        except RuntimeError: print(f"Warning: Could not reshape delta. Original shape: {final_delta.shape}, Target: {(B,C_spec,Freq,Time)}")
    return final_delta

def make_delta_for_transform(input_tensor: torch.Tensor): return compute_deltas_for_transform(input_tensor)
def image_delta_for_transform(x: torch.Tensor): 
    if x.shape[1] != 1: print(f"Warning: image_delta expects 1 input channel, got {x.shape[1]}.")
    delta_1 = make_delta_for_transform(x); delta_2 = make_delta_for_transform(delta_1)
    return torch.cat([x, delta_1, delta_2], dim=1)

def transform_to_spec_srx_pytorch(audio_tensor: torch.Tensor, srx_config_dict: dict) -> np.ndarray:
    # Initialize transforms based on srx_config_dict
    _srx_melspec_transform = AT.MelSpectrogram(
        sample_rate=srx_config_dict['SR'], hop_length=srx_config_dict['hop_length'], 
        n_mels=srx_config_dict['n_mels'], f_min=srx_config_dict['f_min'], 
        f_max=srx_config_dict['f_max'], n_fft=srx_config_dict['n_fft'],
        pad_mode="constant", norm="slaney", onesided=True, mel_scale="htk")
    _srx_db_transform = AT.AmplitudeToDB(stype="power", top_db=srx_config_dict.get('top_db', 80))

    spec = _srx_melspec_transform(audio_tensor.float()) # [B, n_mels, time_frames]
    spec = _srx_db_transform(spec)
    
    norm_type = srx_config_dict.get('normal', 80)
    if norm_type == 80: spec = (spec + 80) / 80
    elif norm_type == 255: spec = spec / 255.0
    else: # MinMax
        spec_min = spec.amin(dim=(-2, -1), keepdim=True); spec_max = spec.amax(dim=(-2, -1), keepdim=True)
        spec = (spec - spec_min) / (spec_max - spec_min + 1e-6)
        
    if spec.ndim == 3: spec = spec.unsqueeze(1) # Add channel dim -> [B, 1, F, T]
    
    # 'in_channels' in srx_config_dict refers to channels for the TIMM encoder part
    apply_delta = srx_config_dict.get('apply_image_delta', False)
    if not apply_delta and srx_config_dict.get('in_channels', 1) == 3 and spec.shape[1] == 1:
        apply_delta = True 
    if apply_delta: spec = image_delta_for_transform(spec) # Output [B, 3, F, T]
        
    return spec.cpu().numpy() # Return NumPy array [B, C_encoder, F, T]


# --- Audio Loading for SeresNext (load_sample_srx_adapted) - Unchanged ---
def load_sample_srx_adapted(path: str, srx_config_dict: dict) -> List[np.ndarray]:
    # ... (implementation remains the same, uses srx_config_dict for SR, duration_train, infer_duration)
    try:
        audio, orig_sr = sf.read(path, dtype="float32", always_2d=False) 
        if audio.ndim > 1: audio = audio.mean(axis=1)
        if orig_sr != srx_config_dict['SR']:
            audio = librosa.resample(y=audio, orig_sr=orig_sr, target_sr=srx_config_dict['SR'])
    except Exception as e_load:
        print(f"Soundfile/Librosa load failed for {path}: {e_load}. Returning empty list.")
        return []
    srx_train_duration = srx_config_dict.get('duration_train', CFG.srx_fallback_config['duration_train'])
    srx_infer_duration = srx_config_dict.get('infer_duration', CFG.srx_fallback_config['infer_duration'])
    target_len_samples = int(srx_config_dict['SR'] * srx_train_duration)
    chunk_len_samples = int(srx_config_dict['SR'] * srx_infer_duration)
    if len(audio) == 0: return []
    num_target_chunks = math.ceil(len(audio) / chunk_len_samples)
    if num_target_chunks == 0 and len(audio) > 0: num_target_chunks = 1
    processed_audio_segments = []
    for i in range(num_target_chunks):
        chunk_center_sample = (i * chunk_len_samples) + (chunk_len_samples // 2)
        start_sample = chunk_center_sample - (target_len_samples // 2)
        end_sample = start_sample + target_len_samples
        pad_start = 0
        if start_sample < 0: pad_start = abs(start_sample); start_sample = 0
        segment = audio[start_sample:end_sample]
        current_len = len(segment)
        final_segment = np.zeros(target_len_samples, dtype=np.float32)
        audio_start_in_final = pad_start
        audio_end_in_final = min(pad_start + current_len, target_len_samples)
        len_to_copy = audio_end_in_final - audio_start_in_final
        if current_len > 0 and len_to_copy > 0:
             final_segment[audio_start_in_final:audio_end_in_final] = segment[:len_to_copy]
        processed_audio_segments.append(final_segment)
    return processed_audio_segments

# --- LIBROSA-BASED PREPROCESSING (Used for EffNet/RegNet - Stage 3) - Unchanged ---
def audio_pad_librosa(audio_data, target_len_samples): # ... (remains same)
    if len(audio_data) >= target_len_samples: return audio_data[:target_len_samples]
    padded_audio = audio_data.copy()
    while len(padded_audio) < target_len_samples:
        needed = target_len_samples - len(padded_audio)
        padded_audio = np.concatenate([padded_audio, audio_data[:needed]])
    return padded_audio[:target_len_samples]

def audio_to_melspec_librosa(audio_data, sr, n_fft, hop_length, n_mels, f_min, f_max, target_shape): # ... (remains same)
    if np.isnan(audio_data).any(): audio_data = np.nan_to_num(audio_data)
    mel_spec = librosa.feature.melspectrogram(y=audio_data, sr=sr, n_fft=n_fft, hop_length=hop_length,
        n_mels=n_mels, fmin=f_min, fmax=f_max, power=2.0, pad_mode="reflect", norm='slaney', htk=True, center=True)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    min_db, max_db = mel_spec_db.min(), mel_spec_db.max()
    if max_db - min_db > 1e-8: mel_spec_norm = (mel_spec_db - min_db) / (max_db - min_db)
    else: mel_spec_norm = np.zeros_like(mel_spec_db)
    if mel_spec_norm.shape != target_shape:
        mel_spec_norm = cv2.resize(mel_spec_norm, target_shape, interpolation=cv2.INTER_LINEAR)
    return mel_spec_norm.astype(np.float32)

def preprocess_audio_librosa(filepath: str, sr_target: int, wav_sec: int, 
                             n_fft: int, hop_length: int, n_mels: int, 
                             f_min: int, f_max: int, target_shape: tuple,
                             input_channels: int) -> Union[np.ndarray, None]: # ... (remains same)
    try: audio_data, sr_orig = librosa.load(filepath, sr=None)
    except Exception as e: print(f"Error loading {filepath} with librosa: {e}"); return None
    if sr_orig != sr_target: audio_data = librosa.resample(audio_data, orig_sr=sr_orig, target_sr=sr_target)
    target_len_samples_per_segment = sr_target * wav_sec
    num_segments = len(audio_data) // target_len_samples_per_segment
    segments_processed = []
    for i in range(num_segments):
        start_sample = i * target_len_samples_per_segment; end_sample = start_sample + target_len_samples_per_segment
        segment_audio = audio_data[start_sample:end_sample]
        if len(segment_audio) < target_len_samples_per_segment:
            segment_audio = audio_pad_librosa(segment_audio, target_len_samples_per_segment)
        mel_spec_norm = audio_to_melspec_librosa(segment_audio, sr_target, n_fft, hop_length,
                                                 n_mels, f_min, f_max, target_shape)
        if input_channels == 1: mel_spec_norm = np.expand_dims(mel_spec_norm, axis=0)
        elif input_channels == 3: mel_spec_norm = np.stack([mel_spec_norm]*3, axis=0)
        segments_processed.append(mel_spec_norm)
    if not segments_processed: return None
    return np.array(segments_processed)


In [4]:

import numpy as np
import openvino.runtime as ov # Ensure it's available
from typing import List, Tuple # For type hinting
import torch # For torch.from_numpy in predict_file_srx_openvino

# These global variables are expected to be defined in Cell 1:
# num_classes (from taxonomy)
# cfg (CFG instance)

# These functions are expected to be defined in Cell 2:
# load_sample_srx_adapted
# transform_to_spec_srx_pytorch

# --- General OpenVINO Inference for Models with Single IR File ---
def run_openvino_infer_on_segments(
    segments_batch_np: np.ndarray, # Shape: [num_segments, C, H, W]
    compiled_model: ov.CompiledModel,
    apply_sigmoid: bool = True
) -> np.ndarray:
    """
    Runs OpenVINO inference segment by segment.
    Assumes compiled_model takes [1, C, H, W] input.
    """
    input_node = compiled_model.inputs[0]
    output_node_port = compiled_model.outputs[0] # Get the output port object
    all_segment_predictions = []

    if segments_batch_np.ndim != 4:
        print(f"Error in run_openvino_infer_on_segments: Expected segments_batch_np to be 4D, got {segments_batch_np.ndim}D")
        global num_classes # Fallback if shape determination fails
        return np.array([np.zeros(num_classes) for _ in range(segments_batch_np.shape[0])])

    # Determine number of classes from the model's output tensor's static part if possible
    model_output_classes = num_classes # Default to global num_classes
    out_partial_shape = output_node_port.partial_shape
    if out_partial_shape.rank.is_static and out_partial_shape.rank.get_length() > 1:
        # Assuming output shape is [Batch, Classes, ...] or [Batch, Classes]
        # The class dimension is typically the second one (index 1)
        if out_partial_shape[1].is_static:
            model_output_classes = out_partial_shape[1].get_length()
        elif out_partial_shape.rank.get_length() == 2 and out_partial_shape[1].is_dynamic: # e.g. [Batch, ?]
             print(f"Warning (run_openvino_infer_on_segments): Class dimension of model output is dynamic. Using global num_classes: {num_classes}")
        # Add more sophisticated checks if needed, e.g. for output [Batch, Time, Classes]
    # else:
    #     print(f"Warning (run_openvino_infer_on_segments): Output rank is dynamic or < 2. Using global num_classes: {num_classes}")


    for i in range(segments_batch_np.shape[0]):
        single_segment_data_np = np.expand_dims(segments_batch_np[i], axis=0)
        single_segment_data_np = single_segment_data_np.astype(input_node.element_type.to_dtype())
        try:
            infer_request = compiled_model.create_infer_request()
            results = infer_request.infer({input_node: single_segment_data_np})
            logits_or_probs_np = results[output_node_port] # Use the port object as key
            
            if apply_sigmoid:
                probabilities_np = 1 / (1 + np.exp(-logits_or_probs_np))
            else:
                probabilities_np = logits_or_probs_np
            all_segment_predictions.append(probabilities_np[0]) 
        except Exception as e:
            print(f"Error during OpenVINO segment inference (idx {i}): {e}")
            all_segment_predictions.append(np.zeros(model_output_classes)) 
    return np.array(all_segment_predictions)


# --- SeresNext Specific OpenVINO Inference ---
def attention_infer_ov_srx(start_idx: int, end_idx: int, 
                           feature_map_np: np.ndarray, # Full feature map: e.g. [1, C_feat, T_feat]
                           compiled_srx_cla: ov.CompiledModel 
                          ) -> np.ndarray:
    """
    Performs inference on a slice of the feature map using the compiled CLA model.
    Outputs max over time of sigmoid(cla_output_slice).
    """
    feat_slice_np = feature_map_np[:, :, start_idx:end_idx]

    global num_classes # Fallback for number of classes
    num_out_classes_srx = num_classes 
    
    cla_output_port = compiled_srx_cla.outputs[0]
    cla_output_partial_shape = cla_output_port.partial_shape

    if cla_output_partial_shape.rank.is_static and cla_output_partial_shape.rank.get_length() > 1:
        # Assuming output is [Batch, Classes, Time_slice] or [Batch, Classes]
        if cla_output_partial_shape[1].is_static:
            num_out_classes_srx = cla_output_partial_shape[1].get_length()
    
    if feat_slice_np.shape[-1] == 0: # If the time dimension of the slice is empty
        # print(f"Warning: feat_slice_np is empty in attention_infer_ov_srx. start:{start_idx}, end:{end_idx}, feat_map_shape:{feature_map_np.shape}")
        return np.zeros((feature_map_np.shape[0], num_out_classes_srx), dtype=np.float32)

    cla_input_node = compiled_srx_cla.inputs[0]
    feat_slice_np_typed = feat_slice_np.astype(cla_input_node.element_type.to_dtype())

    try:
        cla_infer_request = compiled_srx_cla.create_infer_request()
        cla_logits_slice_np = cla_infer_request.infer(
            {cla_input_node: feat_slice_np_typed}
        )[cla_output_port] # Use port object as key
        # Expected cla_logits_slice_np shape: [1, num_out_classes_srx, T_slice]
        
        framewise_pred_sigmoid_np = 1 / (1 + np.exp(-cla_logits_slice_np)) 
        framewise_pred_max_np = framewise_pred_sigmoid_np.max(axis=2) # Max over the time dimension (axis 2)
        return framewise_pred_max_np # Shape: [1, num_out_classes_srx]
    except Exception as e:
        print(f"Error in attention_infer_ov_srx during CLA inference: {e}")
        return np.zeros((feature_map_np.shape[0], num_out_classes_srx), dtype=np.float32)


def predict_file_srx_openvino(
    audio_filepath_str: str,
    srx_loaded_checkpoint_cfg: dict, 
    ov_ef_compiled: ov.CompiledModel,    
    ov_att_compiled: ov.CompiledModel,   # Passed but not used by current attention_infer_ov_srx
    ov_cla_compiled: ov.CompiledModel    
) -> Tuple[List[str], List[np.ndarray]]:
    """
    Orchestrates the multi-part SeresNext OpenVINO inference for a single audio file.
    """
    soundscape_id = Path(audio_filepath_str).stem
    audio_segments_list_np = load_sample_srx_adapted(audio_filepath_str, srx_loaded_checkpoint_cfg)

    if not audio_segments_list_np: 
        print(f"SRX: No audio segments from load_sample for {soundscape_id}. Skipping.")
        return [], []

    all_chunk_final_preds_for_file = []
    row_ids_for_file = []
    
    _tta_delta_srx = cfg.srx_tta_delta 

    for segment_idx, audio_data_for_segment_np in enumerate(audio_segments_list_np):
        audio_tensor_for_spec = torch.from_numpy(audio_data_for_segment_np).unsqueeze(0) 
        spec_np_for_fe = transform_to_spec_srx_pytorch(audio_tensor_for_spec, srx_loaded_checkpoint_cfg)

        fe_input_node = ov_ef_compiled.inputs[0]
        fe_output_port = ov_ef_compiled.outputs[0] # Get port object
        fe_infer_request = ov_ef_compiled.create_infer_request()
        
        try:
            feature_map_from_fe_np = fe_infer_request.infer(
                {fe_input_node: spec_np_for_fe.astype(fe_input_node.element_type.to_dtype())}
            )[fe_output_port] # Use port object as key
        except Exception as e_fe:
            print(f"SRX FE Inference error for {soundscape_id} seg {segment_idx}: {e_fe}")
            # Fallback: create dummy predictions for this segment if FE fails
            global num_classes
            num_preds_to_generate = 1 # Since TTA loop will run once for dummy
            dummy_preds = [np.zeros(num_classes) for _ in range(num_preds_to_generate)]
            all_chunk_final_preds_for_file.extend(dummy_preds) # Add appropriate number of zero preds
            _infer_duration_srx_for_id = srx_loaded_checkpoint_cfg.get('infer_duration', cfg.srx_fallback_config['infer_duration'])
            chunk_end_time_sec = (segment_idx + 1) * _infer_duration_srx_for_id 
            row_ids_for_file.append(f"{soundscape_id}_{chunk_end_time_sec}")
            continue # Move to next segment

        _infer_duration_srx = srx_loaded_checkpoint_cfg.get('infer_duration', cfg.srx_fallback_config['infer_duration'])
        _duration_train_srx = srx_loaded_checkpoint_cfg.get('duration_train', cfg.srx_fallback_config['duration_train'])
        
        feat_time_dim = feature_map_from_fe_np.shape[-1] 
        
        center_ratio = _infer_duration_srx / _duration_train_srx if _duration_train_srx > 0 else 0.5
        window_len_feat = int(feat_time_dim * center_ratio)
        start_feat = (feat_time_dim - window_len_feat) // 2
        end_feat = start_feat + window_len_feat
        
        start_feat = max(0, min(start_feat, feat_time_dim - 1 if feat_time_dim > 0 else 0))
        end_feat = max(start_feat, min(end_feat, feat_time_dim))
        if start_feat >= end_feat and feat_time_dim > 0: # Ensure slice width is at least 1
             end_feat = min(feat_time_dim, start_feat + 1)
        
        pred_center_np = attention_infer_ov_srx(start_feat, end_feat, feature_map_from_fe_np, ov_cla_compiled) 
        tta_preds_list = [pred_center_np]
        
        start_minus = max(0, start_feat - _tta_delta_srx)
        end_minus = max(start_minus, end_feat - _tta_delta_srx) 
        if start_minus >= end_minus and feat_time_dim > 0: end_minus = min(feat_time_dim, start_minus + 1)
        pred_minus_np = attention_infer_ov_srx(start_minus, end_minus, feature_map_from_fe_np, ov_cla_compiled)
        tta_preds_list.append(pred_minus_np)

        start_plus = min(feat_time_dim - 1 if feat_time_dim > 0 else 0, start_feat + _tta_delta_srx) 
        end_plus = min(feat_time_dim, end_feat + _tta_delta_srx)
        if start_plus >= end_plus and feat_time_dim > 0: end_plus = min(feat_time_dim, start_plus + 1)
        pred_plus_np = attention_infer_ov_srx(start_plus, end_plus, feature_map_from_fe_np, ov_cla_compiled)
        tta_preds_list.append(pred_plus_np)
        
        # Ensure all TTA components are [1, num_classes] before squeeze and mean
        squeezed_tta_preds = []
        for p_tta in tta_preds_list:
            if p_tta.ndim > 1:
                squeezed_tta_preds.append(p_tta.squeeze())
            else: # Already 1D
                squeezed_tta_preds.append(p_tta)

        final_chunk_pred_np = (0.5 * squeezed_tta_preds[0] +
                               0.25 * squeezed_tta_preds[1] +
                               0.25 * squeezed_tta_preds[2])
        
        all_chunk_final_preds_for_file.append(final_chunk_pred_np) 
        
        chunk_end_time_sec = (segment_idx + 1) * _infer_duration_srx 
        row_id = f"{soundscape_id}_{chunk_end_time_sec}"
        row_ids_for_file.append(row_id)
        
    return row_ids_for_file, all_chunk_final_preds_for_file



In [5]:

import pandas as pd
import numpy as np
import os
from pathlib import Path
from tqdm.auto import tqdm
import gc

# Expected global variables from Cell 1: cfg, core, species_ids, num_classes
# Expected functions from Cell 2: preprocess_audio_nfnet_dynamic_config, apply_power_to_low_ranked_cols
# Expected functions from Cell 3: run_openvino_infer_on_segments

print("\\n--- Stage 1: NFNet SED Ensemble (OpenVINO) ---")
compiled_nfnet_models = [] # From Cell 1, this should be cfg.compiled_nfnet_models or similar
# For this example, assuming it's loaded here or passed. For now, re-loading for clarity.
# In a real integrated notebook, you'd load models once.
nfnet_model_paths_s1 = [cfg.nfnet_sed_fold0_xml, cfg.nfnet_sed_fold1_xml, cfg.nfnet_sed_fold2_xml]
for path in nfnet_model_paths_s1:
    if not os.path.exists(path):
        print(f"NFNet OV model not found: {path}")
        continue
    try:
        model_ov_nfnet = core.read_model(model=path) # core from Cell 1
        compiled_nfnet_models.append(core.compile_model(model_ov_nfnet, "CPU"))
    except Exception as e:
        print(f"Error loading NFNet model {path} for Stage 1: {e}")

nfnet_predictions_all_files_s1 = []
nfnet_row_ids_all_files_s1 = []

# --- Your Debug/File Listing Logic for Stage 1 ---
current_test_audio_dir_s1 = '/kaggle/input/birdclef-2025/test_soundscapes/'
# Attempt to list files, get basenames without .ogg
try:
    file_basenames_s1 = [f.split('.')[0] for f in sorted(os.listdir(current_test_audio_dir_s1)) if f.endswith('.ogg')]
except FileNotFoundError:
    print(f"Directory not found: {current_test_audio_dir_s1}. Assuming it's empty for debug logic.")
    file_basenames_s1 = []

debug_active_s1 = False
if not file_basenames_s1: # If test_soundscapes is effectively empty
    debug_active_s1 = True
    debug_st_num_s1 = 0   # Start from the beginning for more coverage
    debug_num_s1 = cfg.debug_file_count # Use CFG for count if defined, else default
    
    current_test_audio_dir_s1 = cfg.train_soundscapes_dir # Switch to train_soundscapes
    print(f"NFNet Stage: Test soundscapes empty or not found. Switching to DEBUG MODE.")
    print(f"  Using {current_test_audio_dir_s1} for NFNet stage.")
    try:
        all_train_basenames_s1 = [f.split('.')[0] for f in sorted(os.listdir(current_test_audio_dir_s1)) if f.endswith('.ogg')]
        file_basenames_s1 = all_train_basenames_s1[debug_st_num_s1 : debug_st_num_s1 + debug_num_s1]
    except FileNotFoundError:
        print(f"ERROR: Train soundscapes directory not found for debug: {current_test_audio_dir_s1}")
        file_basenames_s1 = []

print(f"NFNet Stage - Debug mode active: {debug_active_s1}")
print(f"NFNet Stage - Processing {len(file_basenames_s1)} files from {current_test_audio_dir_s1}")
# --- End of Debug/File Listing Logic ---

if compiled_nfnet_models and cfg.nfnet_checkpoint_config and file_basenames_s1:
    for audio_basename_s1 in tqdm(file_basenames_s1, desc="NFNet OV Processing (S1)"):
        audio_filepath_s1 = os.path.join(current_test_audio_dir_s1, audio_basename_s1 + '.ogg')
        if not os.path.exists(audio_filepath_s1):
            print(f"Warning: File {audio_filepath_s1} not found during iteration. Skipping.")
            continue
            
        soundscape_id_s1 = audio_basename_s1 # Already basename
        
        mel_segments_batch_np_s1 = preprocess_audio_nfnet_dynamic_config(audio_filepath_s1, cfg.nfnet_checkpoint_config) 
        
        if mel_segments_batch_np_s1 is None or mel_segments_batch_np_s1.shape[0] == 0:
            print(f"No segments preprocessed for {soundscape_id_s1} in NFNet stage. Skipping file.")
            continue

        file_model_fold_preds_s1 = [] 
        for compiled_model_s1 in compiled_nfnet_models:
            preds_one_fold_s1 = run_openvino_infer_on_segments(mel_segments_batch_np_s1, compiled_model_s1, apply_sigmoid=True)
            file_model_fold_preds_s1.append(preds_one_fold_s1)
        
        if file_model_fold_preds_s1:
            avg_preds_for_file_s1 = np.mean(np.array(file_model_fold_preds_s1), axis=0)
            avg_preds_for_file_pp_s1 = apply_power_to_low_ranked_cols(avg_preds_for_file_s1.copy(), top_k=30, exponent=2)
            nfnet_predictions_all_files_s1.extend(list(avg_preds_for_file_pp_s1))
            
            _wav_sec_nfnet_current = cfg.nfnet_checkpoint_config.get('wav_sec', cfg.nfnet_fallback_config['wav_sec'])
            for i in range(avg_preds_for_file_pp_s1.shape[0]):
                end_time_sec = (i + 1) * _wav_sec_nfnet_current
                nfnet_row_ids_all_files_s1.append(f"{soundscape_id_s1}_{end_time_sec}")
        else:
            print(f"No predictions from NFNet model folds for {soundscape_id_s1}")
else:
    if not compiled_nfnet_models: print("NFNet Stage 1: No OpenVINO models loaded.")
    if not cfg.nfnet_checkpoint_config: print("NFNet Stage 1: Checkpoint config not available.")
    if not file_basenames_s1 : print("NFNet Stage 1: No files to process.")
    print("Skipping main loop of Stage 1 (NFNet OV).")

# Create submission001_ov.csv
if nfnet_row_ids_all_files_s1 and nfnet_predictions_all_files_s1:
    submission_dict_nfnet = {'row_id': nfnet_row_ids_all_files_s1}
    preds_array_nfnet = np.array(nfnet_predictions_all_files_s1)
    
    if preds_array_nfnet.ndim == 2 and preds_array_nfnet.shape[1] == num_classes:
        for i, species_label in enumerate(species_ids):
            submission_dict_nfnet[species_label] = preds_array_nfnet[:, i]
    else:
        print(f"Warning: NFNet output class count ({preds_array_nfnet.shape[1] if preds_array_nfnet.ndim==2 and preds_array_nfnet.size > 0 else 'N/A'}) " +
              f"mismatches global num_classes ({num_classes}). Filling with zeros for submission001_ov.csv.")
        for species_label in species_ids:
            submission_dict_nfnet[species_label] = [0.0] * len(nfnet_row_ids_all_files_s1) if nfnet_row_ids_all_files_s1 else 0.0
            
    df_submission_nfnet = pd.DataFrame(submission_dict_nfnet)
    try:
        sample_sub_df = pd.read_csv(cfg.sample_submission_csv)
        df_submission_nfnet = df_submission_nfnet.reindex(columns=sample_sub_df.columns).fillna(0.0)
    except FileNotFoundError:
        print(f"Warning: Sample submission file not found at {cfg.sample_submission_csv}. Columns may not be perfectly ordered for submission001_ov.csv.")

    df_submission_nfnet.to_csv("submission001_ov.csv", index=False)
    print("Stage 1 (NFNet OV) finished. submission001_ov.csv created/updated.")
else:
    print("Stage 1 (NFNet OV) produced no valid predictions. Creating/leaving empty submission001_ov.csv.")
    try:
        sample_sub_df_s1 = pd.read_csv(cfg.sample_submission_csv)
        empty_preds_nfnet = pd.DataFrame(columns=sample_sub_df_s1.columns)
        empty_preds_nfnet['row_id'] = [] 
        empty_preds_nfnet.to_csv("submission001_ov.csv", index=False)
    except FileNotFoundError:
        print(f"ERROR: Sample submission not found. Cannot create empty submission001_ov.csv with schema.")
gc.collect()


\n--- Stage 1: NFNet SED Ensemble (OpenVINO) ---
NFNet Stage: Test soundscapes empty or not found. Switching to DEBUG MODE.
  Using /kaggle/input/birdclef-2025/train_soundscapes/ for NFNet stage.
NFNet Stage - Debug mode active: True
NFNet Stage - Processing 5 files from /kaggle/input/birdclef-2025/train_soundscapes/


NFNet OV Processing (S1):   0%|          | 0/5 [00:00<?, ?it/s]

Stage 1 (NFNet OV) finished. submission001_ov.csv created/updated.


88

In [6]:

import pandas as pd
import numpy as np
import os
from pathlib import Path
from tqdm.auto import tqdm
import gc

# Expected global variables from Cell 1: cfg, core, species_ids, num_classes
# Expected functions from Cell 2 & 3: predict_file_srx_openvino and its helpers

print("\\n--- Stage 2: Seresnext Model (OpenVINO Multi-Part) ---")
compiled_srx_fe, compiled_srx_att, compiled_srx_cla = None, None, None
srx_models_loaded_successfully = False

if cfg.srx_checkpoint_config is None:
    print("ERROR: SeresNext checkpoint config not loaded! Stage 2 may not function correctly.")

try:
    if os.path.exists(cfg.srx_extract_feature_xml):
        model_fe_srx = core.read_model(model=cfg.srx_extract_feature_xml)
        compiled_srx_fe = core.compile_model(model_fe_srx, "CPU")
    else: print(f"ERROR: SRX FE model not found: {cfg.srx_extract_feature_xml}")
    if os.path.exists(cfg.srx_att_block_att_xml):
        model_att_srx = core.read_model(model=cfg.srx_att_block_att_xml)
        compiled_srx_att = core.compile_model(model_att_srx, "CPU")
    else: print(f"ERROR: SRX ATT model not found: {cfg.srx_att_block_att_xml}")
    if os.path.exists(cfg.srx_att_block_cla_xml):
        model_cla_srx = core.read_model(model=cfg.srx_att_block_cla_xml)
        compiled_srx_cla = core.compile_model(model_cla_srx, "CPU")
    else: print(f"ERROR: SRX CLA model not found: {cfg.srx_att_block_cla_xml}")
        
    if compiled_srx_fe and compiled_srx_att and compiled_srx_cla:
        srx_models_loaded_successfully = True
        print("All Seresnext OpenVINO parts successfully loaded.")
    else:
        print("One or more Seresnext OpenVINO parts failed to load.")
except Exception as e:
    print(f"Exception during loading/compiling Seresnext OpenVINO parts: {e}")

srx_predictions_all_files_s2 = []
srx_row_ids_all_files_s2 = []

# --- Your Debug/File Listing Logic for Stage 2 ---
current_test_audio_dir_s2 = '/kaggle/input/birdclef-2025/test_soundscapes/'
try:
    file_basenames_s2 = [f.split('.')[0] for f in sorted(os.listdir(current_test_audio_dir_s2)) if f.endswith('.ogg')]
except FileNotFoundError:
    print(f"Directory not found: {current_test_audio_dir_s2}. Assuming empty for debug logic.")
    file_basenames_s2 = []

debug_active_s2 = False
if not file_basenames_s2:
    debug_active_s2 = True
    debug_st_num_s2 = 0 
    debug_num_s2 = cfg.debug_file_count
    current_test_audio_dir_s2 = cfg.train_soundscapes_dir
    print(f"SeresNext Stage: Test soundscapes empty. Switching to DEBUG MODE.")
    print(f"  Using {current_test_audio_dir_s2} for SeresNext stage.")
    try:
        all_train_basenames_s2 = [f.split('.')[0] for f in sorted(os.listdir(current_test_audio_dir_s2)) if f.endswith('.ogg')]
        file_basenames_s2 = all_train_basenames_s2[debug_st_num_s2 : debug_st_num_s2 + debug_num_s2]
    except FileNotFoundError:
        print(f"ERROR: Train soundscapes directory not found for debug: {current_test_audio_dir_s2}")
        file_basenames_s2 = []
        
print(f"SeresNext Stage - Debug mode active: {debug_active_s2}")
print(f"SeresNext Stage - Processing {len(file_basenames_s2)} files from {current_test_audio_dir_s2}")
# --- End of Debug/File Listing Logic ---

if srx_models_loaded_successfully and cfg.srx_checkpoint_config and file_basenames_s2:
    for audio_basename_s2 in tqdm(file_basenames_s2, desc="Seresnext OV Processing (S2)"):
        audio_filepath_s2 = os.path.join(current_test_audio_dir_s2, audio_basename_s2 + '.ogg')
        if not os.path.exists(audio_filepath_s2):
            print(f"Warning: File {audio_filepath_s2} not found. Skipping.")
            continue
        try:
            row_ids_curr, preds_curr = predict_file_srx_openvino(
                audio_filepath_s2, cfg.srx_checkpoint_config, 
                compiled_srx_fe, compiled_srx_att, compiled_srx_cla
            )
            srx_row_ids_all_files_s2.extend(row_ids_curr)
            srx_predictions_all_files_s2.extend(preds_curr)
        except Exception as e_file_proc_srx:
            print(f"Error processing file {audio_filepath_s2} in SeresNext Stage 2: {e_file_proc_srx}")
else:
    if not srx_models_loaded_successfully: print("SeresNext Stage 2: Not all OV parts loaded.")
    if not cfg.srx_checkpoint_config: print("SeresNext Stage 2: Checkpoint config not available.")
    if not file_basenames_s2: print("SeresNext Stage 2: No files to process.")
    print("Skipping main loop of Stage 2 (SeresNext OV).")

# Create submission002_ov.csv
if srx_row_ids_all_files_s2 and srx_predictions_all_files_s2:
    submission_dict_srx = {'row_id': srx_row_ids_all_files_s2}
    preds_array_srx_raw = np.array(srx_predictions_all_files_s2)
    
    srx_model_output_num_classes = 0
    if preds_array_srx_raw.ndim == 2 and preds_array_srx_raw.shape[0] > 0:
        srx_model_output_num_classes = preds_array_srx_raw.shape[1]
    elif preds_array_srx_raw.ndim == 1 and preds_array_srx_raw.size > 0 : # Single segment result
        srx_model_output_num_classes = preds_array_srx_raw.shape[0]
        preds_array_srx_raw = np.expand_dims(preds_array_srx_raw, axis=0)

    if srx_model_output_num_classes > 0:
        for i, global_species_label in enumerate(species_ids):
            if i < srx_model_output_num_classes:
                submission_dict_srx[global_species_label] = preds_array_srx_raw[:, i]
            else:
                submission_dict_srx[global_species_label] = 0.0
        
        df_submission_srx = pd.DataFrame(submission_dict_srx)
        try:
            sample_sub_df = pd.read_csv(cfg.sample_submission_csv)
            df_submission_srx = df_submission_srx.reindex(columns=sample_sub_df.columns).fillna(0.0)
        except FileNotFoundError:
            print(f"Warning: Sample submission file not found at {cfg.sample_submission_csv}. Columns may not be perfectly ordered for submission002_ov.csv.")
        df_submission_srx.to_csv("submission002_ov.csv", index=False)
        print("Stage 2 (Seresnext OV) finished. submission002_ov.csv created/updated.")
    else:
        print("Stage 2 (Seresnext OV) produced predictions, but array format is unexpected or empty.")
        # Create empty placeholder logic as in Stage 1
        try:
            sample_sub_df_s2 = pd.read_csv(cfg.sample_submission_csv)
            empty_preds_srx = pd.DataFrame(columns=sample_sub_df_s2.columns); empty_preds_srx['row_id'] = []
            empty_preds_srx.to_csv("submission002_ov.csv", index=False)
            print("Created empty submission002_ov.csv due to prediction format issue.")
        except FileNotFoundError: print(f"ERROR: Sample submission not found. Cannot create empty submission002_ov.csv.")
else:
    print("Stage 2 (Seresnext OV) produced no valid predictions. Creating/leaving empty submission002_ov.csv.")
    try:
        sample_sub_df_s2 = pd.read_csv(cfg.sample_submission_csv)
        empty_preds_srx = pd.DataFrame(columns=sample_sub_df_s2.columns); empty_preds_srx['row_id'] = []
        empty_preds_srx.to_csv("submission002_ov.csv", index=False)
    except FileNotFoundError: print(f"ERROR: Sample submission not found. Cannot create empty submission002_ov.csv.")
gc.collect()


\n--- Stage 2: Seresnext Model (OpenVINO Multi-Part) ---
All Seresnext OpenVINO parts successfully loaded.
SeresNext Stage: Test soundscapes empty. Switching to DEBUG MODE.
  Using /kaggle/input/birdclef-2025/train_soundscapes/ for SeresNext stage.
SeresNext Stage - Debug mode active: True
SeresNext Stage - Processing 5 files from /kaggle/input/birdclef-2025/train_soundscapes/


Seresnext OV Processing (S2):   0%|          | 0/5 [00:00<?, ?it/s]

Stage 2 (Seresnext OV) finished. submission002_ov.csv created/updated.


19

In [7]:

import pandas as pd 
import numpy as np  
import os          
from pathlib import Path 
from tqdm.auto import tqdm 
import gc 

# Expected global variables from Cell 1: cfg, core, species_ids, num_classes
# Expected functions from Cell 2: preprocess_audio_librosa
# Expected functions from Cell 3: run_openvino_infer_on_segments

print("\\n--- Stage 3: EfficientNet B0 + RegNetY_008 Ensemble (OpenVINO) ---")
compiled_effnet_model = None
compiled_regnet_model = None
effreg_models_loaded_count = 0 

try:
    if os.path.exists(cfg.efficientnet_b0_xml):
        model_eff_ov = core.read_model(model=cfg.efficientnet_b0_xml) # core from Cell 1
        compiled_effnet_model = core.compile_model(model_eff_ov, "CPU")
        effreg_models_loaded_count += 1
        print(f"  Successfully loaded & compiled EfficientNet B0.")
    else: print(f"Warning: EfficientNet B0 OV model not found at {cfg.efficientnet_b0_xml}")
    if os.path.exists(cfg.regnety_008_xml):
        model_reg_ov = core.read_model(model=cfg.regnety_008_xml) # core from Cell 1
        compiled_regnet_model = core.compile_model(model_reg_ov, "CPU")
        effreg_models_loaded_count += 1
        print(f"  Successfully loaded & compiled RegNetY_008.")
    else: print(f"Warning: RegNetY_008 OV model not found at {cfg.regnety_008_xml}")
    if effreg_models_loaded_count > 0: print(f"Successfully loaded {effreg_models_loaded_count} models for Stage 3.")
    else: print("No models loaded for Stage 3 (EfficientNet/RegNet).")
except Exception as e:
    print(f"Exception during loading/compiling Stage 3 OpenVINO models: {e}")

effreg_predictions_all_files_s3 = []
effreg_row_ids_all_files_s3 = []

# --- Your Debug/File Listing Logic for Stage 3 ---
current_test_audio_dir_s3 = '/kaggle/input/birdclef-2025/test_soundscapes/'
try:
    file_basenames_s3 = [f.split('.')[0] for f in sorted(os.listdir(current_test_audio_dir_s3)) if f.endswith('.ogg')]
except FileNotFoundError:
    print(f"Directory not found: {current_test_audio_dir_s3}. Assuming empty for debug logic.")
    file_basenames_s3 = []

debug_active_s3 = False
if not file_basenames_s3:
    debug_active_s3 = True
    debug_st_num_s3 = 0
    debug_num_s3 = cfg.debug_file_count 
    current_test_audio_dir_s3 = cfg.train_soundscapes_dir
    print(f"Stage 3: Test soundscapes empty. Switching to DEBUG MODE.")
    print(f"  Using {current_test_audio_dir_s3} for Stage 3.")
    try:
        all_train_basenames_s3 = [f.split('.')[0] for f in sorted(os.listdir(current_test_audio_dir_s3)) if f.endswith('.ogg')]
        file_basenames_s3 = all_train_basenames_s3[debug_st_num_s3 : debug_st_num_s3 + debug_num_s3]
    except FileNotFoundError:
        print(f"ERROR: Train soundscapes directory not found for debug: {current_test_audio_dir_s3}")
        file_basenames_s3 = []

print(f"Stage 3 - Debug mode active: {debug_active_s3}")
print(f"Stage 3 - Processing {len(file_basenames_s3)} files from {current_test_audio_dir_s3}")
# --- End of Debug/File Listing Logic ---

if effreg_models_loaded_count > 0 and file_basenames_s3:
    for audio_basename_s3 in tqdm(file_basenames_s3, desc="EffNet/RegNet OV Processing (S3)"):
        audio_filepath_s3 = os.path.join(current_test_audio_dir_s3, audio_basename_s3 + '.ogg')
        if not os.path.exists(audio_filepath_s3):
            print(f"Warning: File {audio_filepath_s3} not found. Skipping.")
            continue
            
        soundscape_id_s3 = audio_basename_s3
        
        mel_segments_batch_np_effreg = preprocess_audio_librosa(
            audio_filepath_s3, cfg.sr_effreg, cfg.wav_sec_effreg, cfg.n_fft_effreg, 
            cfg.hop_length_effreg, cfg.n_mels_effreg, cfg.f_min_effreg, 
            cfg.f_max_effreg, cfg.effreg_target_shape, cfg.effreg_input_channels
        )
        
        if mel_segments_batch_np_effreg is None or mel_segments_batch_np_effreg.shape[0] == 0:
            print(f"No segments for {soundscape_id_s3} in Stage 3. Skipping file.")
            continue

        file_model_preds_for_this_stage_s3 = [] 
        if compiled_effnet_model:
            preds_effnet = run_openvino_infer_on_segments(mel_segments_batch_np_effreg, compiled_effnet_model, apply_sigmoid=True)
            file_model_preds_for_this_stage_s3.append(preds_effnet)
        if compiled_regnet_model:
            preds_regnet = run_openvino_infer_on_segments(mel_segments_batch_np_effreg, compiled_regnet_model, apply_sigmoid=True)
            file_model_preds_for_this_stage_s3.append(preds_regnet)
        
        if file_model_preds_for_this_stage_s3: 
            avg_preds_for_file_effreg = np.mean(np.array(file_model_preds_for_this_stage_s3), axis=0)
            effreg_predictions_all_files_s3.extend(list(avg_preds_for_file_effreg))
            for i in range(avg_preds_for_file_effreg.shape[0]):
                end_time_sec = (i + 1) * cfg.wav_sec_effreg
                effreg_row_ids_all_files_s3.append(f"{soundscape_id_s3}_{end_time_sec}")
        else:
            print(f"No predictions from EffNet/RegNet models for file {soundscape_id_s3}")
else:
    if effreg_models_loaded_count <= 0 : print("Stage 3: No EffNet/RegNet OV models loaded.")
    if not file_basenames_s3 : print("Stage 3: No files to process.")
    print("Skipping main loop of Stage 3 (EffNet/RegNet OV).")

# Create submission003_ov.csv
if effreg_row_ids_all_files_s3 and effreg_predictions_all_files_s3:
    submission_dict_effreg = {'row_id': effreg_row_ids_all_files_s3}
    preds_array_effreg_raw = np.array(effreg_predictions_all_files_s3)
    
    effreg_model_output_num_classes = 0
    if preds_array_effreg_raw.ndim == 2 and preds_array_effreg_raw.shape[0] > 0:
        effreg_model_output_num_classes = preds_array_effreg_raw.shape[1]
    elif preds_array_effreg_raw.ndim == 1 and preds_array_effreg_raw.size > 0 :
        effreg_model_output_num_classes = preds_array_effreg_raw.shape[0]
        preds_array_effreg_raw = np.expand_dims(preds_array_effreg_raw, axis=0)

    if effreg_model_output_num_classes > 0:
        for i, global_species_label in enumerate(species_ids): 
            if i < effreg_model_output_num_classes:
                submission_dict_effreg[global_species_label] = preds_array_effreg_raw[:, i]
            else:
                submission_dict_effreg[global_species_label] = 0.0
        
        df_submission_effreg = pd.DataFrame(submission_dict_effreg)
        try:
            sample_sub_df = pd.read_csv(cfg.sample_submission_csv)
            df_submission_effreg = df_submission_effreg.reindex(columns=sample_sub_df.columns).fillna(0.0)
        except FileNotFoundError:
            print(f"Warning: Sample submission file not found at {cfg.sample_submission_csv}. Columns may not be perfectly ordered for submission003_ov.csv.")
            
        df_submission_effreg.to_csv("submission003_ov.csv", index=False)
        print("Stage 3 (EffNet/RegNet OV) finished. submission003_ov.csv created/updated.")
    else:
        print("Stage 3 (EffNet/RegNet OV) produced predictions, but array format is unexpected or empty.")
        try:
            sample_sub_df_s3 = pd.read_csv(cfg.sample_submission_csv)
            empty_preds_effreg = pd.DataFrame(columns=sample_sub_df_s3.columns); empty_preds_effreg['row_id'] = []
            empty_preds_effreg.to_csv("submission003_ov.csv", index=False)
            print("Created empty submission003_ov.csv due to prediction format issue.")
        except FileNotFoundError: print(f"ERROR: Sample submission not found. Cannot create empty submission003_ov.csv.")
else:
    print("Stage 3 (EffNet/RegNet OV) produced no valid predictions. Creating/leaving empty submission003_ov.csv.")
    try:
        sample_sub_df_s3 = pd.read_csv(cfg.sample_submission_csv)
        empty_preds_effreg = pd.DataFrame(columns=sample_sub_df_s3.columns); empty_preds_effreg['row_id'] = []
        empty_preds_effreg.to_csv("submission003_ov.csv", index=False)
    except FileNotFoundError: print(f"ERROR: Sample submission not found. Cannot create empty submission003_ov.csv.")
gc.collect()


\n--- Stage 3: EfficientNet B0 + RegNetY_008 Ensemble (OpenVINO) ---
  Successfully loaded & compiled EfficientNet B0.
  Successfully loaded & compiled RegNetY_008.
Successfully loaded 2 models for Stage 3.
Stage 3: Test soundscapes empty. Switching to DEBUG MODE.
  Using /kaggle/input/birdclef-2025/train_soundscapes/ for Stage 3.
Stage 3 - Debug mode active: True
Stage 3 - Processing 5 files from /kaggle/input/birdclef-2025/train_soundscapes/


EffNet/RegNet OV Processing (S3):   0%|          | 0/5 [00:00<?, ?it/s]

Stage 3 (EffNet/RegNet OV) finished. submission003_ov.csv created/updated.


76211

In [8]:
sub=pd.read_csv("submission001_ov.csv")
sub.head()

,row_id,1139490,1192948,1194042,126247,1346504,134933,135045,1462711,1462737,...,yebfly1,yebsee1,yecspi2,yectyr1,yehbla2,yehcar1,yelori1,yeofly1,yercac1,ywcpar
0,H02_20230420_074000_5,0.265243,0.265278,0.267786,0.279686,0.264836,0.272068,0.271525,0.268027,0.565969,...,0.283845,0.281809,0.530803,0.282779,0.261651,0.299595,0.274230,0.288304,0.269076,0.546852
1,H02_20230420_074000_10,0.268379,0.266536,0.266006,0.275890,0.266216,0.270557,0.272299,0.272799,0.599278,...,0.280307,0.283543,0.525831,0.279434,0.261005,0.290165,0.274085,0.288358,0.268376,0.550685
2,H02_20230420_074000_15,0.269628,0.265606,0.267346,0.271370,0.265243,0.274576,0.280475,0.268414,0.568162,...,0.277212,0.277956,0.532141,0.278003,0.273805,0.290671,0.273854,0.286139,0.268229,0.519075
3,H02_20230420_074000_20,0.269608,0.267661,0.270204,0.273917,0.266150,0.268590,0.276041,0.266143,0.579530,...,0.280839,0.275097,0.531365,0.274740,0.261057,0.296788,0.269622,0.285011,0.260826,0.512211
4,H02_20230420_074000_25,0.264345,0.266268,0.265091,0.268856,0.263396,0.268610,0.275265,0.265941,0.572356,...,0.278549,0.279030,0.523685,0.282198,0.263358,0.299707,0.270199,0.282689,0.265380,0.518463


In [9]:
sub_1=pd.read_csv("submission002_ov.csv")
sub_1.head()

,row_id,1139490,1192948,1194042,126247,1346504,134933,135045,1462711,1462737,...,yebfly1,yebsee1,yecspi2,yectyr1,yehbla2,yehcar1,yelori1,yeofly1,yercac1,ywcpar
0,H02_20230420_074000_5,0.004895,0.005957,0.002549,0.007454,0.005169,0.010527,0.008595,0.005301,0.008339,...,0.006165,0.007043,0.006247,0.017187,0.006013,0.012932,0.006091,0.014329,0.008525,0.008032
1,H02_20230420_074000_10,0.004339,0.004665,0.002804,0.006800,0.004141,0.008109,0.007407,0.005252,0.007515,...,0.006312,0.005313,0.006658,0.010072,0.004775,0.009904,0.006842,0.011145,0.006389,0.005712
2,H02_20230420_074000_15,0.004790,0.004545,0.003618,0.012026,0.005535,0.008828,0.008899,0.003815,0.007394,...,0.007846,0.006863,0.006609,0.012162,0.005824,0.013752,0.008661,0.012999,0.011015,0.008997
3,H02_20230420_074000_20,0.008338,0.010247,0.004884,0.006503,0.014545,0.008240,0.007790,0.008504,0.011363,...,0.009731,0.005555,0.006034,0.018568,0.005750,0.015813,0.010828,0.013556,0.012814,0.012792
4,H02_20230420_074000_25,0.005202,0.005184,0.002156,0.009145,0.005150,0.009059,0.008351,0.005960,0.008037,...,0.006909,0.005781,0.004085,0.012737,0.004998,0.010771,0.006598,0.011121,0.009107,0.005354


In [10]:
sub_2=pd.read_csv("submission003_ov.csv")
sub_2.head()

,row_id,1139490,1192948,1194042,126247,1346504,134933,135045,1462711,1462737,...,yebfly1,yebsee1,yecspi2,yectyr1,yehbla2,yehcar1,yelori1,yeofly1,yercac1,ywcpar
0,H02_20230420_074000_5,0.000005,0.000027,0.000892,0.000690,0.000678,0.000749,0.000078,0.000031,0.000107,...,0.001794,0.000769,0.000829,0.000315,0.000110,0.002596,0.000992,0.004791,0.000940,0.000372
1,H02_20230420_074000_10,0.000003,0.000009,0.001314,0.000850,0.000377,0.000581,0.000052,0.000014,0.000042,...,0.001981,0.000647,0.000478,0.000230,0.000036,0.001805,0.000377,0.004570,0.001246,0.000446
2,H02_20230420_074000_15,0.000010,0.000049,0.000288,0.000814,0.000235,0.002213,0.000093,0.000062,0.000141,...,0.002535,0.002305,0.001698,0.000492,0.000219,0.005216,0.001090,0.004691,0.001124,0.000489
3,H02_20230420_074000_20,0.000012,0.000074,0.000455,0.000166,0.000161,0.005532,0.000154,0.000059,0.000169,...,0.003433,0.001760,0.001828,0.000799,0.000380,0.011803,0.002768,0.006846,0.001918,0.001740
4,H02_20230420_074000_25,0.000006,0.000038,0.000444,0.000477,0.000233,0.000492,0.000034,0.000035,0.000106,...,0.002662,0.000707,0.001303,0.000464,0.000129,0.004282,0.001376,0.003643,0.001181,0.001234


In [11]:
import pandas as pd
import os
import gc # Good to have for potentially large dataframes

# --- Blending Configuration ---
# Paths to your intermediate OpenVINO submission files
submission_paths = [
    "/kaggle/working/submission001_ov.csv",  # Corresponds to weight 0.67 (e.g., NFNet Ensemble)
    "/kaggle/working/submission002_ov.csv",  # Corresponds to weight 0.05 (e.g., SeresNext)
    "/kaggle/working/submission003_ov.csv"   # Corresponds to weight 0.33 (e.g., EffNet/RegNet Ensemble)
]
# Your specified weights, ensure order matches submission_paths above
# Weight for submission001_ov.csv = 0.67
# Weight for submission002_ov.csv = 0.05
# Weight for submission003_ov.csv = 0.33
blending_weights = [0.05, 0.60, 0.35] 

# Final output path
final_submission_path = "submission.csv"

# --- Get Target Bird Species Columns ---
# This logic is taken directly from your reference
list_TARGETs = []
try:
    train_audio_dir_for_labels = '/kaggle/input/birdclef-2025/train_audio/'
    if os.path.exists(train_audio_dir_for_labels) and os.path.isdir(train_audio_dir_for_labels):
        # Assuming species names are directory names
        list_TARGETs = sorted([d for d in os.listdir(train_audio_dir_for_labels) 
                               if os.path.isdir(os.path.join(train_audio_dir_for_labels, d))])
        if not list_TARGETs: 
            print("Warning: train_audio directory has no subdirectories.")
    else:
        print(f"Warning: Directory '{train_audio_dir_for_labels}' not found for deriving target labels.")

    if not list_TARGETs: # Fallback if directory method fails or yields no targets
        print("Attempting to infer columns from first submission file.")
        if os.path.exists(submission_paths[0]):
            temp_df_for_cols = pd.read_csv(submission_paths[0])
            list_TARGETs = sorted([col for col in temp_df_for_cols.columns if col != 'row_id'])
        else:
            raise FileNotFoundError(f"Fallback failed: {submission_paths[0]} not found.")
            
except Exception as e:
    print(f"Error determining target columns: {e}. Please define list_TARGETs manually or check paths.")
    # As a last resort, you might need to hardcode list_TARGETs if all else fails
    # For now, let this error propagate if critical.
    # list_TARGETs = [...] # Manual definition
    if not list_TARGETs: # If still not defined after fallbacks
        raise SystemExit("Could not determine target species columns for blending.")

if not list_TARGETs:
    raise SystemExit("Target species list (list_TARGETs) is empty. Cannot proceed with blending.")

print(f"Blending for {len(list_TARGETs)} target species. First 5: {list_TARGETs[:5]}")

# --- Load and Prepare DataFrames (Renaming with Suffixes) ---
dataframes_renamed = [] 
all_suffixed_columns_to_delete = [] # To keep track for later deletion

for i, path in enumerate(submission_paths):
    if not os.path.exists(path):
        # Option 1: Raise an error if a file is crucial
        raise FileNotFoundError(f"Submission file not found: {path}. Cannot proceed with blending.")
        # Option 2: Create a dummy df with zeros if a file might be missing but you want to proceed
        # print(f"Warning: Submission file not found: {path}. Creating a dummy with zeros.")
        # temp_df_dummy = pd.read_csv(submission_paths[0 if i > 0 else 1]) # Use another df for row_ids
        # df = temp_df_dummy[['row_id']].copy()
        # for target_col in list_TARGETs:
        #     df[target_col] = 0.0
        
    df = pd.read_csv(path)
    print(f"Loaded {path}, shape: {df.shape}")
    
    current_suffixed_targets = [f'{TARGET}_{i}' for TARGET in list_TARGETs]
    all_suffixed_columns_to_delete.extend(current_suffixed_targets)
    
    rename_dict = {original_col: suffixed_col for original_col, suffixed_col in zip(list_TARGETs, current_suffixed_targets)}
    
    # Check if all original target columns exist in df before renaming
    # and add them with 0 if missing (as per your reference's implication)
    for original_col in list_TARGETs:
        if original_col not in df.columns:
            print(f"Warning: Column '{original_col}' not found in {path}. Adding it with 0.0 values.")
            df[original_col] = 0.0 
            
    df_renamed = df.rename(columns=rename_dict)
    dataframes_renamed.append(df_renamed)

# --- Merge DataFrames ---
# Start with the first renamed dataframe
dfs_merged = dataframes_renamed[0]

# Sequentially merge the rest
# Your reference merged df0 and df1. We adapt for three (or more).
print("\nMerging dataframes...")
for i in range(1, len(dataframes_renamed)):
    # Select only 'row_id' and the suffixed target columns from the current dataframe to merge
    # This avoids duplicate non-target columns if they exist.
    suffixed_cols_for_current_df = [f'{TARGET}_{i}' for TARGET in list_TARGETs]
    cols_to_merge_from_current_df = ['row_id'] + suffixed_cols_for_current_df
    
    # Ensure the dataframe being merged actually has these columns (it should after renaming)
    df_to_merge = dataframes_renamed[i][cols_to_merge_from_current_df]
    
    dfs_merged = pd.merge(dfs_merged, df_to_merge, on='row_id', how='inner') 
    # Using 'inner' merge: only rows present in ALL submissions will be kept.
    # This is critical. If one submission has fewer rows, the output will be based on that smallest set.

print(f"Shape of merged DataFrame before blending: {dfs_merged.shape}")
if dfs_merged.empty:
    raise SystemExit("Merged DataFrame is empty. Check if 'row_id's match across all submission files and if 'inner' merge is appropriate.")

# --- Perform Weighted Blending ---
print("\nPerforming weighted blending...")
for i_target, target_species in enumerate(list_TARGETs):
    # Initialize a Series/column for the blended values for this target_species
    blended_column_series = pd.Series(0.0, index=dfs_merged.index)
    
    for model_idx in range(len(dataframes_renamed)): # Iterate through 0, 1, 2 for three models
        suffixed_col_name = f'{target_species}_{model_idx}' # e.g. birdX_0, birdX_1, birdX_2
        
        # Ensure the suffixed column exists (it should after merging)
        if suffixed_col_name in dfs_merged.columns:
            blended_column_series += dfs_merged[suffixed_col_name] * blending_weights[model_idx]
        else:
            print(f"Warning: Suffixed column {suffixed_col_name} not found in merged df during blending. This shouldn't happen.")
            
    dfs_merged[target_species] = blended_column_series

# --- Delete Intermediate Suffixed Columns ---
print("\nDeleting intermediate suffixed columns...")
for col_to_delete in all_suffixed_columns_to_delete:
    if col_to_delete in dfs_merged.columns:
        del dfs_merged[col_to_delete]
gc.collect()

# --- Prepare Final Submission File ---
# At this point, dfs_merged contains 'row_id' and the final blended target columns
# (which have the original species names).
dfs_final = dfs_merged

# Save the blended submission
dfs_final.to_csv(final_submission_path, index=False)
print(f"\nBlended submission saved to: {final_submission_path}")
print(f"Final submission shape: {dfs_final.shape}")

# Optional: Display head of the final submission
print("\nHead of the final blended submission:")
print(dfs_final.head())

Blending for 206 target species. First 5: ['1139490', '1192948', '1194042', '126247', '1346504']
Loaded /kaggle/working/submission001_ov.csv, shape: (60, 207)
Loaded /kaggle/working/submission002_ov.csv, shape: (60, 207)
Loaded /kaggle/working/submission003_ov.csv, shape: (60, 207)

Merging dataframes...
Shape of merged DataFrame before blending: (60, 619)

Performing weighted blending...

Deleting intermediate suffixed columns...

Blended submission saved to: submission.csv
Final submission shape: (60, 207)

Head of the final blended submission:
                   row_id   1139490   1192948   1194042    126247   1346504  \
0   H02_20230420_074000_5  0.016201  0.016847  0.015231  0.018698  0.016581   
1  H02_20230420_074000_10  0.016023  0.016129  0.015443  0.018172  0.015927   
2  H02_20230420_074000_15  0.016359  0.016024  0.015639  0.021069  0.016665   
3  H02_20230420_074000_20  0.018488  0.019557  0.016600  0.017656  0.022091   
4  H02_20230420_074000_25  0.016341  0.016437  0.014